In [ ]:
repository_filter: list[str] = []
metric: str = "cyclomaticComplexity"

In [ ]:
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import warnings

warnings.simplefilter("ignore")

df = dt.read_csv("../samples/method_quality_metrics.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)

In [ ]:
if len(df) == 0 or metric not in df.columns:
    fig = qu.empty_figure(
        f"No data available (metric='{metric}')."
    )
else:
    # Compute per-repo stats
    repo_stats = df.groupby("repoShort")[metric].agg(
        ["median", "std", "count", "max"]
    ).reset_index()
    repo_stats.columns = ["repo", "median", "std", "count", "max"]
    repo_stats = repo_stats[repo_stats["count"] >= 5]  # need enough data points
    repo_stats["std"] = repo_stats["std"].fillna(0)
    repo_stats["cv"] = repo_stats["std"] / (repo_stats["median"] + 1e-9)  # coefficient of variation

    total_repos = len(repo_stats)
    n_show = min(5, total_repos)

    if total_repos <= 25:
        # Small enough to show all
        selected = repo_stats.sort_values("median", ascending=False)
        gap_label = None
    else:
        # Pick the most interesting repos:
        # - Top 3 by median (worst complexity)
        worst = repo_stats.nlargest(10, "median")
        # - Top 2 by coefficient of variation (most divergent — long tails)
        divergent = repo_stats[~repo_stats["repo"].isin(worst["repo"])].nlargest(5, "cv")
        # - Bottom 3 by median (healthiest)
        best = repo_stats[
            ~repo_stats["repo"].isin(worst["repo"]) &
            ~repo_stats["repo"].isin(divergent["repo"])
        ].nsmallest(10, "median")

        import pandas as pd
        selected = pd.concat([worst, divergent, best])
        elided = total_repos - len(selected)
        gap_label = f"··· {elided} more repos ···"

    # Sort selected: worst first, then divergent, then best
    selected = selected.sort_values("median", ascending=False)

    # Normalize medians to 0-1 for color mapping
    med_min = selected["median"].min()
    med_max = selected["median"].max()
    med_range = med_max - med_min if med_max > med_min else 1

    # Cap outliers so violin shapes are visible
    cap = df[metric].quantile(0.95)

    fig = go.Figure()

    # Add worst and divergent repos
    repos_added = 0
    for _, row in selected.iterrows():
        repo = row["repo"]
        repo_data = df[df["repoShort"] == repo][metric].clip(upper=cap)
        norm = (row["median"] - med_min) / med_range
        r = int(76 + norm * (239 - 76))
        g = int(175 + norm * (83 - 175))
        b = int(80 + norm * (80 - 80))
        color = f"rgb({r},{g},{b})"

        # Tag divergent repos
        is_divergent = gap_label and row["cv"] > selected["cv"].quantile(0.7)
        label = f"{repo} {'(high variance)' if is_divergent and norm > 0.3 else ''}"

        fig.add_trace(
            go.Violin(
                y=repo_data,
                name=label.strip(),
                box_visible=True,
                meanline_visible=True,
                points="outliers",
                fillcolor=color,
                line_color="#333",
                opacity=0.75,
            )
        )
        repos_added += 1

        # Insert gap indicator after the divergent repos (roughly in the middle)
        if gap_label and repos_added == len(selected) - 3:
            fig.add_trace(
                go.Violin(
                    y=[selected["median"].median()],
                    name=gap_label,
                    fillcolor="#E0E0E0",
                    line_color="#E0E0E0",
                    opacity=0.3,
                    box_visible=False,
                    meanline_visible=False,
                    points=False,
                )
            )

    metric_label = metric.replace("_", " ").title()
    subtitle = f"Showing {len(selected)} most interesting of {total_repos} repositories" if gap_label else f"All {total_repos} repositories"
    fig.update_layout(
        title=dict(
            text=f"Quality Distribution: {metric_label}<br><sup>{subtitle} — worst, most variable, and healthiest — outliers capped at 95th percentile</sup>",
            font=dict(size=16),
        ),
        yaxis_title=metric_label,
        showlegend=False,
        height=650,
        width=1200,
        plot_bgcolor="white",
        margin=dict(l=60, r=40, t=80, b=140),
        yaxis=dict(range=[0, cap * 1.1]),
        xaxis=dict(tickangle=-35, tickfont=dict(size=10)),
    )
fig.show()